# Seance 7 — Univariee : histogramme, barplot, boxplot

Voir user guide :
- Specifying Data : https://altair-viz.github.io/user_guide/data.html
- Encodings : https://altair-viz.github.io/user_guide/encodings/index.html
- Marks (1/3) : https://altair-viz.github.io/user_guide/marks/index.html

## Objectifs
- Choisir le bon graphique selon Q / O / N
- Utiliser **pandas** pour preparer les tableaux (comptages, bins, etc.)
- Produire des figures avec titre, sous-titre, source

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

data_url = "https://raw.githubusercontent.com/datamisc/ts-2024/main/data.csv"
df = pd.read_csv(data_url, compression="gzip", low_memory=False)

## 1) Histogramme (Q) : age

Variable : `V241458x` (age).

Ici, on construit l'histogramme avec **pandas** :
- creer des intervalles (bins)
- compter le nombre de repondants par intervalle

Ensuite on trace un barplot a partir de ce tableau de comptage.

In [ ]:
age = df.loc[df['V241458x'] > 0, ['V241458x']].copy()

bins = list(range(18, 91, 5)) + [100]
age['age_bin'] = pd.cut(age['V241458x'], bins=bins, right=False)
hist_age = (age.groupby('age_bin', as_index=False).size().rename(columns={'size': 'n'}))

hist_age.head()

In [ ]:
alt.Chart(hist_age).mark_bar(color='#3B82F6').encode(
    x=alt.X('age_bin', type='ordinal', title="Tranches d'age (5 ans)"),
    y=alt.Y('n', type='quantitative', title='Nombre de repondants')
).properties(
    title=alt.TitleParams(
        text="Distribution de l'age des repondants",
        subtitle=[
            "On observe une forte concentration entre 45 et 75 ans.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=650,
    height=280
)

### Hack-Time 1 (10 min)

Changez les bins :
- bins de 10 ans
- bins de 2 ans

Puis re-tracez le graphique. Que change le message ?

In [ ]:
# Hack-Time 1 : votre code ici

## 2) Barplot (O) : education

Variable : `V241200` (education, categories 1-5).

On construit d'abord le tableau de comptage avec pandas, puis on trace un barplot.

In [ ]:
edu = (
    df.loc[df['V241200'] > 0, ['V241200']]
    .assign(education=lambda d: d['V241200'].astype(int))
    .groupby('education', as_index=False)
    .size()
    .rename(columns={'size': 'n'})
)

edu

In [ ]:
alt.Chart(edu).mark_bar(color='#10B981').encode(
    x=alt.X('education', type='ordinal', title="Niveau d'education (code 1-5)"),
    y=alt.Y('n', type='quantitative', title='Nombre de repondants')
).properties(
    title=alt.TitleParams(
        text="Niveau d'education dans l'echantillon",
        subtitle=[
            "Les categories 2-5 dominent, ce qui peut influencer les analyses.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=260
)

### Hack-Time 2 (10 min)

Recoder les categories d'education en 3 groupes lisibles :
- "Sans diplome/secondaire"
- "Superieur"
- "Autre/NSP" (si vous en avez)

Puis refaites le barplot.

In [ ]:
# Hack-Time 2 : votre code ici

## 3) Barplot (O) : ideologie

Variable : `V241177` (ideologie 1-7) + une categorie speciale (ex: 99).

On filtre 1-7 pour travailler sur l'echelle ideologique.

In [ ]:
ideo = (
    df.loc[df['V241177'].between(1, 7), ['V241177']]
    .assign(ideologie=lambda d: d['V241177'].astype(int))
    .groupby('ideologie', as_index=False)
    .size()
    .rename(columns={'size': 'n'})
)

alt.Chart(ideo).mark_bar(color='#F59E0B').encode(
    x=alt.X('ideologie', type='ordinal', title='Ideologie (1-7)'),
    y=alt.Y('n', type='quantitative', title='Nombre de repondants')
).properties(
    title=alt.TitleParams(
        text="Auto-placement ideologique (1-7)",
        subtitle=[
            "Le centre (4) est la modalite la plus frequente.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=260
)

## 4) Boxplot (Q) : thermometre Harris

Variable : `V241156` (0-100).

Pour respecter la regle 'pandas d'abord', on calcule les statistiques (quartiles) avec pandas puis on dessine un boxplot avec des couches simples (barre + regles).

In [ ]:
therm = df.loc[df['V241156'].between(0, 100), ['V241156']].copy()
q1 = therm['V241156'].quantile(0.25)
q3 = therm['V241156'].quantile(0.75)
median = therm['V241156'].median()
vmin = therm['V241156'].min()
vmax = therm['V241156'].max()

box = pd.DataFrame([{
    'groupe': 'Tous',
    'q1': float(q1),
    'q3': float(q3),
    'median': float(median),
    'min': float(vmin),
    'max': float(vmax),
}])

box

In [ ]:
box_rect = alt.Chart(box).mark_bar(size=40, color='#6366F1').encode(
    x=alt.X('groupe', type='nominal', title=''),
    y=alt.Y('q1', type='quantitative', title='Thermometre Harris (0-100)', scale=alt.Scale(domain=[0, 100])),
    y2=alt.Y2('q3', type='quantitative')
)

whisker = alt.Chart(box).mark_rule(color='black').encode(
    x=alt.X('groupe', type='nominal'),
    y=alt.Y('min', type='quantitative'),
    y2=alt.Y2('max', type='quantitative')
)

med = alt.Chart(box).mark_rule(color='white', strokeWidth=3).encode(
    x=alt.X('groupe', type='nominal'),
    y=alt.Y('median', type='quantitative')
)

alt.layer(whisker, box_rect, med).properties(
    title=alt.TitleParams(
        text="Distribution des evaluations de Kamala Harris",
        subtitle=[
            "La moitie des repondants se situe entre q1 et q3 (boite violette).",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=260,
    height=380
)

### Hack-Time 3 (10 min)

Refaites le meme boxplot pour `V241157` (thermometre Trump).
- gardez l'echelle 0-100
- changez les couleurs
- ecrivez un bon titre et un bon sous-titre

In [ ]:
# Hack-Time 3 : votre code ici

## Hack-Time 4 (15 min) — Mini-projet univarie

Choisissez une variable univariee pertinente en comportement politique, par exemple :
- `V241201` : interet pour la politique
- `V241258` : environnement vs entreprises
- `V241042` : intention d'aller voter

Et produisez un graphique qui respecte :
- titre explicite
- sous-titre avec un exemple de lecture
- source
- mise en forme lisible

In [ ]:
# Hack-Time 4 : votre graphique ici